### 获取结构化结果范式

#### 1. with_structured_output

In [1]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="评分(10分制)")
    
# 当加了 include_raw=True, 输出结果中就会包含原始的 AIMessage
structured_model = model.with_structured_output(Movie, include_raw=True)
response = structured_model.invoke("帮我介绍一个星级穿越这个电影")

rprint(type(response))
rprint(response)  

<class 'dict'>

{
    'raw': AIMessage(
        content='{ "title": "星际穿越介绍", "year": 2014, "director": "克里斯托弗·诺兰", "rating": 9.4 }',
        additional_kwargs={
            'parsed': Movie(title='星际穿越介绍', year=2014, director='克里斯托弗·诺兰', rating=9.4),
            'refusal': None
        },
        response_metadata={
            'token_usage': {
                'completion_tokens': 1164,
                'prompt_tokens': 19,
                'total_tokens': 1183,
                'completion_tokens_details': {
                    'accepted_prediction_tokens': None,
                    'audio_tokens': None,
                    'reasoning_tokens': 1118,
                    'rejected_prediction_tokens': None
                },
                'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}
            },
            'model_provider': 'openai',
            'model_name': 'glm-5.3',
            'system_fingerprint': None,
            'id': '202611121040064a37bc2e868944d5',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--01a1cda0-6e82-7ec0-a41a-33e601557460-0',
        tool_calls=[],
        invalid_tool_calls=[],
        usage_metadata={
            'input_tokens': 19,
            'output_tokens': 1164,
            'total_tokens': 1183,
            'input_token_details': {'cache_read': 0},
            'output_token_details': {'reasoning': 1118}
        }
    ),
    'parsed': Movie(title='星际穿越介绍', year=2014, director='克里斯托弗·诺兰', rating=9.4),
    'parsing_error': None
}

#### 2. 使用输出解析器(不推荐)

这种方法更传统, 依赖于在提示词中明确指示模型输出特定格式的文本, 然后使用解析器进行转换
其流程是: 提示词指导(引导生成指定类型) -> 模型生成文本 -> 解析器转换

In [5]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from rich import print as rprint
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

# 1. 创建提示词模版
prompt_template = ChatPromptTemplate.from_messages([
    ("system", "回答用户问题, 必须始终输出一个包含 title(电影标题)和year(上映年份)的 JSON 对象"),
    ("human", "问题: {question}")
])

# 2. 模型初始化
# 从 .env 文件中加载环境变量
load_dotenv(override=True)
ZHIPUAI_API_KEY = os.getenv("ZHIPUAI_API_KEY")
ZHIPUAI_API_BASE_URL = os.getenv("ZHIPUAI_API_BASE_URL")

# 获取大模型
model = init_chat_model(
    model='glm-5.3',
    model_provider="openai",
    api_key=ZHIPUAI_API_KEY,
    base_url=ZHIPUAI_API_BASE_URL
)

# 3. 定义结构
class Movie(BaseModel):
    """电影信息"""
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")

# 4. 创建输出解析器
parser = JsonOutputParser(pydantic_object=Movie)

# 5. 创建链
chain = prompt_template | model | parser

# 6. 调用(返回字典)
response = chain.invoke({"question": "介绍电影 盗梦空间"})

rprint(response)

{'title': '盗梦空间', 'year': 2010}